# Logística y gestión de inventario: reparto de carga en camiones — *versión Tema 4*

Este cuaderno explica una **versión mejorada** del problema que ya resolviste en el [Tema 3](https://github.com/jjcastroschez/Introduccion-Computacion-Cientifica/blob/main/03_variables_tipos_simples/ejemplos/camiones_exp_py.ipynb), aprovechando los nuevos conceptos del Tema 4: **bucles**, **condicionales** y **manejo de excepciones**.

## ¿Qué hacía la versión del Tema 3?

El programa del Tema 3 resolvía la ecuación diofántica $ax + by = r$ asumiendo que **alguien** (el gestor de la flota) le había hecho previamente los cálculos en papel. En concreto, el usuario tenía que:

1. **Verificar a mano** que la **Identidad de Bézout** se cumplía: que mcd($a$, $b$) divide a $r$.
2. **Calcular a mano** el **inverso modular** de $a$ módulo $b$, e introducirlo como un dato más.
3. **Confiar** en que las soluciones obtenidas eran números naturales (es decir, no negativos).

## ¿Qué mejora la versión del Tema 4?

Con lo aprendido, vamos a hacer que **sea el ordenador quien haga todo eso**. El usuario solo introduce los datos del enunciado original (las capacidades de los dos vehículos y la carga total), y el programa se encarga de:

1. **Calcular mcd($a$, $b$)** automáticamente, usando el [algoritmo de Euclides](euclides_icc.ipynb).
2. **Verificar la identidad de Bézout** con una sentencia condicional.
3. **Buscar el inverso modular** con un bucle `for` que prueba candidatos.
4. **Comprobar** que las soluciones obtenidas son números naturales.
5. **Validar la entrada** del usuario con `try-except` para que el programa no se rompa nunca.

## Paso 1: Lectura validada de los datos

Antes hacíamos simplemente `int(input(...))`. Eso bastaba para introducir el problema, pero **el programa se rompía** si el usuario tecleaba mal. Ahora, con `try-except` y un bucle `while`, podemos pedir el dato hasta que sea válido.

También aprovechamos para añadir **validaciones lógicas**: las capacidades y la carga han de ser positivas; además, la capacidad del vehículo pequeño debe ser menor que la del grande.

In [ ]:
entrada_valida = False
while not entrada_valida:
    try:
        capacidad_mg = int(input('Dame la capacidad del vehículo más grande: '))
        capacidad_mp = int(input('Dame la capacidad del vehículo más pequeño: '))
        carga = int(input('Dame la carga que quieres distribuir: '))
    except ValueError:
        print('  ⚠️ Tienes que introducir números enteros.')
    else:
        if capacidad_mp > 0 and capacidad_mg > capacidad_mp and carga > 0:
            entrada_valida = True
        else:
            print('  ⚠️ Tienes que introducir números enteros positivos, además la capacidad del primer camión debe ser mayor que el segundo.')

> 💡 **Observa**: en el Tema 3 también pedíamos un cuarto dato, el **inverso modular**, que el usuario tenía que calcular previamente. Esa entrada **ya no la pedimos**: la calculará el programa más abajo. Además, mira como el `try-except-else` se usa hasta obtener valores válidos, una vez conseguidos ejecutamos el código fuera del `while`🚀

## Paso 2: Calcular mcd($a$, $b$) por el algoritmo de Euclides

La identidad de Bézout dice que $ax + by = r$ tiene soluciones enteras si y solo si **mcd($a$, $b$) divide a $r$**. Antes de calcular nada, debemos verificarlo. Y para eso necesitamos primero el mcd.

Aplicamos el bucle `while` del algoritmo de Euclides que viste en el ejemplo [euclides_exp_py.ipynb](./euclides_exp_py.ipynb):

In [ ]:
a = capacidad_mg    # copiamos los valores para no perder los originales
b = capacidad_mp

while b != 0:
    resto = a % b
    a = b
    b = resto

mcd = a   # tras el bucle, el último divisor distinto de cero es el mcd

print(f'mcd({capacidad_mg}, {capacidad_mp}) = {mcd}')

## Paso 3: Verificar la identidad de Bézout

Con el mcd ya calculado, una sola sentencia `if` nos basta para verificar si la ecuación tiene solución entera:

In [ ]:
if carga % mcd != 0:
    print(f'❌ La identidad de Bézout NO se cumple: {mcd} no divide a {carga}.')
    print('   No existen soluciones enteras para repartir la carga.')
    hay_solucion = False
else:
    print(f'✅ La identidad de Bézout se cumple: {mcd} divide a {carga}.')
    hay_solucion = True

> 🎯 **Esta línea es importante**: a partir de aquí, todo el resto del programa solo tiene sentido si `hay_solucion` es verdadero. Vamos a envolverlo todo en un gran `if`.

## Paso 4: Calcular el inverso modular automáticamente

Recordemos qué es el **inverso modular** de $a$ módulo $b$: es el único $k \in \{1, 2, \ldots, b-1\}$ tal que:

$$ a \cdot k \equiv 1 \pmod{b} $$

Es decir, el resto de dividir $a \cdot k$ entre $b$ vale exactamente $1$. En el Tema 3 el usuario tenía que calcularlo a mano. Pero ahora, con un **bucle `for`**, podemos probar todos los candidatos desde $1$ hasta $b-1$ y quedarnos con el primero que cumple la condición. ¡Es búsqueda directa!

In [ ]:
if hay_solucion:
    inverso = 0
    for k in range(1, capacidad_mp):
        if (capacidad_mg * k) % capacidad_mp == 1:
            inverso = k
            break    # encontrado, salimos del bucle

    if inverso == 0:
        print(f'❌ No existe inverso de {capacidad_mg} módulo {capacidad_mp}.')
    else:
        print(f'Inverso de {capacidad_mg} módulo {capacidad_mp} = {inverso}')

> 💡 **Observación matemática**: si mcd($a$, $b$) = 1, el inverso de $a$ módulo $b$ **siempre existe**. Como en nuestra tabla del Tema 3 todos los casos tenían mcd 1, el bucle siempre encontrará un inverso. La comprobación `if inverso == 0` es una **red de seguridad** para casos generales (por ejemplo, si mcd > 1).

## Paso 5: Calcular las soluciones $x$ e $y$

Una vez tenemos el inverso, los cálculos son **idénticos a los del Tema 3**:

$$ x = (r \cdot \text{inverso}) \bmod b $$
$$ y = \frac{r - a \cdot x}{b} $$

In [ ]:
if hay_solucion and inverso != 0:
    num_vehiculos_g = (carga * inverso) % capacidad_mp
    num_vehiculos_p = (carga - capacidad_mg * num_vehiculos_g) // capacidad_mp
    print(f'Resultado de la ecuación: x = {num_vehiculos_g}, y = {num_vehiculos_p}')

## Paso 6: Comprobar que las soluciones son **naturales**

¡Aquí está la mejora más importante! La identidad de Bézout garantiza soluciones **enteras**, pero las soluciones físicamente válidas son las **naturales** ($\geq 0$): no podemos enviar $-3$ camiones. Con un simple `if` lo verificamos:

In [ ]:
if hay_solucion and inverso != 0:
    if num_vehiculos_g >= 0 and num_vehiculos_p >= 0:
        total = num_vehiculos_g * capacidad_mg + num_vehiculos_p * capacidad_mp
        print(f'\n✅ Solución encontrada:')
        print(f'   {num_vehiculos_g} vehículos de capacidad {capacidad_mg}')
        print(f'   {num_vehiculos_p} vehículos de capacidad {capacidad_mp}')
        print(f'   Total transportado: {total} (debe coincidir con la carga {carga})')
    else:
        print(f'\n⚠️ La ecuación tiene solución entera, pero NO con números naturales:')
        print(f'   x = {num_vehiculos_g}, y = {num_vehiculos_p}')
        print('   No se puede repartir la carga con estos vehículos sin dejar')
        print('   mercancía en tierra o desperdiciar espacio.')

## 🧪 Pruébalo con los casos del Tema 3

Ahora solo necesitas introducir tres datos (capacidades y carga) — **el inverso lo calcula el programa**:

| Capacidad Mayor | Capacidad Menor | Carga | Resultado esperado |
|:-:|:-:|:-:|:--|
| 7  | 5 | 31 | 3 grandes, 2 pequeños |
| 9  | 7 | 44 | 1 grande, 5 pequeños |
| 9  | 7 | 41 | 3 grandes, 2 pequeños |
| 13 | 5 | 41 | 2 grandes, 3 pequeños |
| 5  | 3 | 37 | 2 grandes, 9 pequeños |
| 4  | 3 | 37 | 1 grande, 11 pequeños |

Y prueba también algún caso que **falle**:

| Capacidad Mayor | Capacidad Menor | Carga | Qué debería detectar |
|:-:|:-:|:-:|:--|
| 6  | 4 | 31 | mcd(6, 4) = 2 no divide a 31 |
| 7  | 5 | 3  | Solución entera pero no natural |
| hola | -3 | abc | Validación de entrada |

## 🎯 Conceptos del Tema 4 que has practicado

* ✅ **Bucle `while`** para el algoritmo de Euclides (no sabemos cuántas vueltas hará falta).
* ✅ **Bucle `for in range`** combinado con **`break`** para buscar el inverso modular.
* ✅ **Condicionales encadenadas** (`if-else`) para distinguir entre los tres posibles resultados (sin solución, solución entera pero no natural, solución válida).
* ✅ **Combinación de `try-except` con `while`** para validar la entrada del usuario.
* ✅ **Reescritura constructivista**: hemos partido del programa del Tema 3 y, sin cambiar la matemática de fondo, hemos hecho que el ordenador haga **el trabajo que antes le pedíamos al usuario**.

## 🚀 Para reflexionar

* La búsqueda del inverso modular por **fuerza bruta** (probando todos los candidatos) es razonable cuando $b$ es pequeño. ¿Qué pasaría si $b$ fuera, digamos, 1 000 000? Existe un método más eficiente: el **algoritmo extendido de Euclides**, que calcula a la vez el mcd y los coeficientes de Bézout. Lo verás en cursos de Álgebra o Análisis Numérico.
* En este problema solo hemos buscado **una** solución particular. Pero las ecuaciones diofánticas suelen tener **infinitas soluciones enteras**. Concretamente, si $(x_0, y_0)$ es una solución de $ax + by = r$, entonces para cualquier entero $t$, $(x_0 + bt, y_0 - at)$ también lo es. ¿Cómo modificarías el programa para listar todas las soluciones con $x, y \geq 0$? (Pista: necesitarás otro bucle.)